# SignalStack Backtesting

Evaluate Flow Score signals against historical price data.

**Goal:** measure hit rate of high vs low Flow Scores against forward 7-day returns.

Replace the mock historical data with real cached snapshots from `engine/signal_engine.py` once you have collected enough days of `history_<asset>` cache entries.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime

## 1. Load or generate historical data

In [ ]:
rng = np.random.default_rng(42)
n_days = 180
historical_data = pd.DataFrame({
    'date': pd.date_range(end=datetime.utcnow(), periods=n_days, freq='D'),
    'flow_score': rng.uniform(20, 85, n_days),
    'btc_price': np.cumsum(rng.standard_normal(n_days) * 500) + 45000,
})
historical_data.head()

## 2. Compute forward 7-day returns

In [ ]:
historical_data['return_7d'] = (
    historical_data['btc_price'].shift(-7) / historical_data['btc_price'] - 1
) * 100
historical_data.dropna(subset=['return_7d']).head()

## 3. Hit rates

In [ ]:
high_score = historical_data[historical_data['flow_score'] > 70].dropna(subset=['return_7d'])
low_score = historical_data[historical_data['flow_score'] < 30].dropna(subset=['return_7d'])

high_win_rate = (high_score['return_7d'] > 0).mean() if len(high_score) else float('nan')
low_accuracy = (low_score['return_7d'] < 0).mean() if len(low_score) else float('nan')

print(f'High Flow Score (>70) Win Rate: {high_win_rate:.2%}')
print(f'Low Flow Score (<30) Accuracy: {low_accuracy:.2%}')
print(f'Avg 7d return after high signal:  {high_score["return_7d"].mean():.2f}%')
print(f'Avg 7d return after low signal:   {low_score["return_7d"].mean():.2f}%')

## 4. Visualize

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=historical_data['date'],
    y=historical_data['flow_score'],
    name='Flow Score',
    yaxis='y1',
    line=dict(color='#00D084'),
))
fig.add_trace(go.Scatter(
    x=historical_data['date'],
    y=historical_data['btc_price'],
    name='BTC Price',
    yaxis='y2',
    line=dict(color='#9CA3AF'),
))
fig.update_layout(
    title='Flow Score vs BTC Price',
    yaxis=dict(title='Flow Score'),
    yaxis2=dict(title='Price (USD)', overlaying='y', side='right'),
    hovermode='x unified',
    paper_bgcolor='#0F1419',
    plot_bgcolor='#0F1419',
    font=dict(color='white'),
)
fig.show()